# NB_lab_test_trends

Builds `adb_caaresync.gold.lab_test_trends` — one row per test type per calendar month with test volumes, abnormal/critical rates, and month-over-month trend via LAG. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caaresync.gold.lab_test_trends AS

WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                               AS year,
    MONTH(lr.test_date)                                              AS month,
    COUNT(*)                                                         AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
  FROM adb_caaresync.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

enriched AS (
  SELECT
    ma.test_name,
    ma.year,
    ma.month,
    c.month_name,
    CONCAT('Q', c.quarter)                                           AS quarter_name,
    ma.total_tests,
    ma.abnormal_test_count,
    ma.critical_test_count,
    CAST(ma.abnormal_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))
                                                                     AS abnormal_rate_pct,
    CAST(ma.critical_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))
                                                                     AS critical_rate_pct
  FROM      monthly_agg                      ma
  LEFT JOIN adb_caaresync.silver.calendar     c
         ON c.date_key = CAST(DATE_FORMAT(DATE_TRUNC('MONTH', MAKE_DATE(ma.year, ma.month, 1)), 'yyyyMMdd') AS INT)
)

SELECT
  e.*,
  LAG(e.abnormal_rate_pct) OVER (PARTITION BY e.test_name ORDER BY e.year, e.month)
                                                                     AS prev_month_abnormal_rate_pct,
  CAST(
    e.abnormal_rate_pct
    - LAG(e.abnormal_rate_pct) OVER (PARTITION BY e.test_name ORDER BY e.year, e.month)
    AS DECIMAL(6,2)
  )                                                                  AS abnormal_rate_change_pct

FROM enriched e

In [0]:
record_count = spark.table("adb_caaresync.gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))